In [49]:
import pandas as pd
import numpy as np
from web3 import Web3
import json
from dotenv import load_dotenv
import os
from hexbytes import HexBytes
load_dotenv()
ALC_RPC_URL = os.getenv("ALC_RPC_URL")
w3 = Web3(Web3.HTTPProvider(ALC_RPC_URL))

In [2]:
df = pd.read_parquet("../data/raw/univ3_pool_logs.parquet")

In [3]:
df.head()

,address,topics,data,transaction_index,log_index,transaction_hash,block_number,block_hash,block_timestamp,date,last_modified
0,0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8,[0xc42079f94a6350d7e6235f29174924f928cc2ac818e...,0xffffffffffffffffffffffffffffffffffffffffffff...,150,327,0x313f88a08cbda72eca653433f60d35ac6044597fe03c...,15985296,0x4ace3df61aaa8fc1e8e1021a9456da32a9ffe5b99d4a...,2022-11-16 21:34:59,2022-11-16,2022-11-17 01:31:26.350131
1,0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8,[0x7a53080ba414158be7ec69b987b5fb7d07dee101fe8...,0x000000000000000000000000c36442b4a4522e871399...,42,104,0xf2117efba6da137734545b19bfc58c25cdd5d75ef4a8...,15985300,0xc66b6799821636b0f11148f45394d3add9316ff6c6e5...,2022-11-16 21:35:47,2022-11-16,2022-11-17 01:31:29.168530
2,0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,0x00000000000000000000000000000000000000000000...,137,201,0x7fb8fe34835a9bed2abd9194fe16ba2d4035e32703e5...,15985331,0xb11a520b5e0d92e21fb2f1b81ce61bcf82c0a6dd02b8...,2022-11-16 21:41:59,2022-11-16,2022-11-17 01:31:53.364683
3,0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8,[0x70935338e69775456a85ddef226c395fb668b63fa01...,0x000000000000000000000000c36442b4a4522e871399...,137,204,0x7fb8fe34835a9bed2abd9194fe16ba2d4035e32703e5...,15985331,0xb11a520b5e0d92e21fb2f1b81ce61bcf82c0a6dd02b8...,2022-11-16 21:41:59,2022-11-16,2022-11-17 01:31:53.364711
4,0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,0x00000000000000000000000000000000000000000000...,98,171,0x3b71fdfd9b24e64f5665c3393b40eeeffddddfaa78ed...,15985333,0xe7c98e94fc66e43db84ee4dcf68c95529d3852cab3a8...,2022-11-16 21:42:23,2022-11-16,2022-11-17 01:31:54.467517


In [5]:
df.info

<bound method DataFrame.info of                                            address  \
0       0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   
1       0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   
2       0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   
3       0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   
4       0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   
...                                            ...   
679673  0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   
679674  0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   
679675  0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   
679676  0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   
679677  0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8   

                                                   topics  \
0       [0xc42079f94a6350d7e6235f29174924f928cc2ac818e...   
1       [0x7a53080ba414158be7ec69b987b5fb7d07dee101fe8...   
2       [0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...   
3       [0x70935338e69775456a85ddef226c395fb668b63fa01...   
4       [0x0c3

## Need to decode the transactions

In [42]:
abi = json.load(open("../data/abis/UniswapV3Pool.json"))["abi"]
address = w3.to_checksum_address("8ad599c3a0ff1de082011efddc58f1908eb6e6d8")
pool = w3.eth.contract(address=address, abi=abi)

In [51]:
def make_log(row):
    topics = row["topics"]
    if isinstance(topics, str):
        topics = ast.literal_eval(topics)
    topics = [HexBytes(t) for t in topics]

    data = row["data"]
    if isinstance(data, str):
        data = HexBytes(data)
    tx_hash = row["transaction_hash"]
    if isinstance(tx_hash, str):
        tx_hash = HexBytes(tx_hash)

    block_hash = row["block_hash"]
    if isinstance(block_hash, str):
        block_hash = HexBytes(block_hash)

    return {
        "address": Web3.to_checksum_address(row["address"]),
        "topics": topics,
        "data": data,
        "blockNumber": int(row["block_number"]),
    "transactionHash": tx_hash,
    "transactionIndex": int(row["transaction_index"]),
    "logIndex": int(row["log_index"]),
    "blockHash": block_hash,
    }

In [52]:
def decode_log(row):
    log = make_log(row)
    for event in [pool.events.Swap, pool.events.Mint, pool.events.Burn, pool.events.Collect]:
        ev = event()
        if hasattr(ev, "process_log"):
            fn = ev.process_log
        else:
            fn = ev.processLog
        try:
            return fn(log)
        except Exception:
            continue
    return None

In [53]:
def decode_flat(row):
    res = decode_log(row)
    if res is None: return None
    args = dict(res["args"])
    out = {**args}
    out["event"] = res["event"]
    out["address"] = res["address"]
    out["block_number"] = res["blockNumber"]
    out["block_timestamp"] = row["block_timestamp"]
    out["tx_hash"] = res["transactionHash"].hex()
    out["log_index"] = res["logIndex"]
    out["tx_index"] = res["transactionIndex"]
    return out

In [55]:
decoded_series = df.apply(decode_flat, axis=1)

In [56]:
print(decoded_series)

0         {'sender': '0xE592427A0AEce92De3Edee1F18E0157C...
1         {'owner': '0xC36442b4a4522E871399CD717aBDD847A...
2         {'owner': '0xC36442b4a4522E871399CD717aBDD847A...
3         {'owner': '0xC36442b4a4522E871399CD717aBDD847A...
4         {'owner': '0xC36442b4a4522E871399CD717aBDD847A...
                                ...                        
679673    {'sender': '0xEf1c6E67703c7BD7107eed8303Fbe6EC...
679674    {'sender': '0xEf1c6E67703c7BD7107eed8303Fbe6EC...
679675    {'owner': '0xC36442b4a4522E871399CD717aBDD847A...
679676    {'sender': '0xEf1c6E67703c7BD7107eed8303Fbe6EC...
679677    {'sender': '0xEf1c6E67703c7BD7107eed8303Fbe6EC...
Length: 679678, dtype: object


In [57]:
decoded_rows = [r for r in decoded_series if r is not None]

In [58]:
decoded_df = pd.DataFrame(decoded_rows)

In [59]:
decoded_df.head()

,sender,recipient,amount0,amount1,sqrtPriceX96,liquidity,tick,event,address,block_number,block_timestamp,tx_hash,log_index,tx_index,owner,tickLower,tickUpper,amount
0,0xE592427A0AEce92De3Edee1F18E0157C05861564,0xE592427A0AEce92De3Edee1F18E0157C05861564,-49540580434,41079000000000000000,2278156445805771936898597302158205,11457725834822228820,205341.0,Swap,0x8ad599c3A0ff1De082011EFDDc58f1908eb6e6D8,15985296,2022-11-16 21:34:59,313f88a08cbda72eca653433f60d35ac6044597fe03cae...,327,150,NaN,NaN,NaN,NaN
1,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,5416821,4616252638023322,NaN,NaN,NaN,Mint,0x8ad599c3A0ff1De082011EFDDc58f1908eb6e6D8,15985300,2022-11-16 21:35:47,f2117efba6da137734545b19bfc58c25cdd5d75ef4a8df...,104,42,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,196200.0,214140.0,437629118975
2,NaN,NaN,0,0,NaN,NaN,NaN,Burn,0x8ad599c3A0ff1De082011EFDDc58f1908eb6e6D8,15985331,2022-11-16 21:41:59,7fb8fe34835a9bed2abd9194fe16ba2d4035e32703e55b...,201,137,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,194820.0,208620.0,0
3,NaN,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,198462304,209663943994578563,NaN,NaN,NaN,Collect,0x8ad599c3A0ff1De082011EFDDc58f1908eb6e6D8,15985331,2022-11-16 21:41:59,7fb8fe34835a9bed2abd9194fe16ba2d4035e32703e55b...,204,137,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,194820.0,208620.0,NaN
4,NaN,NaN,0,0,NaN,NaN,NaN,Burn,0x8ad599c3A0ff1De082011EFDDc58f1908eb6e6D8,15985333,2022-11-16 21:42:23,3b71fdfd9b24e64f5665c3393b40eeeffddddfaa78ed6e...,171,98,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,197400.0,211260.0,0


In [63]:
bigint_cols = [
    "amount0",
    "amount1",
    "sqrtPriceX96",
    "liquidity",
    "amount",
]

for c in bigint_cols:
    if c in decoded_df.columns:
        decoded_df[c] = decoded_df[c].astype("string")

In [64]:
decoded_df.to_parquet("../data/processed/decoded_pool_transactions.parquet")